# Urban Step 1: Raw Image Data Collection Notebook

This notebook allows you to view the **live ROS Camera stream** (`/csi_cam_0/image_raw`) on JetRacer and **capture raw image frames** at intersections, traffic signs, pedestrian crosswalks, and lanes into urban dataset folders (`urban_dataset_A`, `urban_dataset_B`, etc.).

### 1. Setup Environment & ROS Node

In [ ]:
import os
import sys
import cv2
import glob
import uuid
import base64
import threading
import numpy as np
from pathlib import Path

# Add parent directory to sys.path
parent_dir = Path.cwd().parent.parent
if str(parent_dir) not in sys.path:
    sys.path.append(str(parent_dir))

import rospy
from sensor_msgs.msg import Image as ROSImage
try:
    from jetracer.utils import bgr8_to_jpeg
except ImportError:
    from utils import bgr8_to_jpeg

# Initialize ROS Node
try:
    rospy.init_node('urban_data_collection_notebook', anonymous=True, disable_signals=True)
    print("[+] ROS Node initialized successfully!")
except Exception as e:
    print(f"[*] ROS Node notice: {e}")


### 2. Live Camera Stream & Capture UI

In [ ]:
import ipywidgets
from IPython.display import display

DATASETS = ['A', 'B', 'C']
latest_ros_image = None
_lock = threading.Lock()

# Unregister previous ROS subscriber if active
if 'ros_sub' in globals() and ros_sub is not None:
    try:
        ros_sub.unregister()
    except Exception:
        pass

dataset_widget = ipywidgets.Dropdown(options=DATASETS, description='Urban Dataset', value='A')
count_widget   = ipywidgets.IntText(description='Image Count', value=0, disabled=True)
capture_button = ipywidgets.Button(description='Capture Urban Image', button_style='success', icon='camera', layout=ipywidgets.Layout(width='220px', height='40px'))
status_widget  = ipywidgets.HTML(value="<p style='color:green;'><b>Ready to capture urban images.</b></p>")

camera_html_widget = ipywidgets.HTML(
    value="<p><b>Waiting for ROS Camera Topic...</b></p>",
    layout=ipywidgets.Layout(width='240px', height='240px')
)
snapshot_widget = ipywidgets.Image(format='jpeg', width=224, height=224)

def get_dataset_dir():
    d_dir = os.path.join(Path.cwd(), f"urban_dataset_{dataset_widget.value}")
    os.makedirs(d_dir, exist_ok=True)
    return d_dir

def refresh_count():
    d_dir = get_dataset_dir()
    images = glob.glob(os.path.join(d_dir, "*.jpg"))
    count_widget.value = len(images)

refresh_count()

def on_dataset_change(change):
    refresh_count()
    status_widget.value = f"<p style='color:blue;'>Switched to Urban Dataset <b>{change['new']}</b> ({count_widget.value} images saved).</p>"

dataset_widget.observe(on_dataset_change, names='value')

def capture_image_callback(b):
    global latest_ros_image
    if latest_ros_image is not None:
        d_dir = get_dataset_dir()
        img_name = f"urban_{str(uuid.uuid1())[:8]}.jpg"
        save_path = os.path.join(d_dir, img_name)
        cv2.imwrite(save_path, latest_ros_image)
        
        refresh_count()
        snapshot_widget.value = bgr8_to_jpeg(latest_ros_image)
        status_widget.value = f"<p style='color:green;'><b>[+] Saved Raw Urban Image #{count_widget.value}</b> -> {img_name}</p>"
    else:
        status_widget.value = "<p style='color:red;'><b>[!] No ROS camera frame received yet!</b></p>"

capture_button.on_click(capture_image_callback)

def ros_image_to_cv2(msg):
    im = np.frombuffer(msg.data, dtype=np.uint8).reshape(msg.height, msg.width, -1)
    if msg.encoding in ['rgb8', 'rgb8']:
        im = cv2.cvtColor(im, cv2.COLOR_RGB2BGR)
    elif msg.encoding == 'rgba8':
        im = cv2.cvtColor(im, cv2.COLOR_RGBA2BGR)
    elif msg.encoding == 'bgra8':
        im = cv2.cvtColor(im, cv2.COLOR_BGRA2BGR)
    if im.shape[0] != 224 or im.shape[1] != 224:
        im = cv2.resize(im, (224, 224))
    return im

def camera_callback(msg):
    global latest_ros_image
    if not _lock.acquire(blocking=False):
        return
    try:
        latest_ros_image = ros_image_to_cv2(msg)
        jpeg_bytes = bgr8_to_jpeg(latest_ros_image)
        b64 = base64.b64encode(jpeg_bytes).decode('utf-8')
        
        html_str = f'''
        <div style="font-family: monospace; background: #1e1e1e; padding: 6px; border-radius: 6px; display: inline-block;">
            <h5 style="margin:0 0 4px 0; color: #ffffff;">Urban Live Camera Feed</h5>
            <img src="data:image/jpeg;base64,{b64}" style="width:224px; height:224px; border:2px solid #00ff00; border-radius:4px;" />
        </div>
        '''
        camera_html_widget.value = html_str
    except Exception:
        pass
    finally:
        try:
            _lock.release()
        except RuntimeError:
            pass

topic_name = "/csi_cam_0/image_raw"
ros_sub = rospy.Subscriber(topic_name, ROSImage, camera_callback, queue_size=1, buff_size=2**24)

ui_layout = ipywidgets.VBox([
    ipywidgets.HBox([camera_html_widget, snapshot_widget]),
    dataset_widget,
    count_widget,
    capture_button,
    status_widget
])

display(ui_layout)
print(f"[*] Subscribed to ROS Camera Topic: {topic_name}")
